**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/encorescience/encore-registry", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/rozsa-2023"], check=True)
    os.chdir("/content/encore/entries/rozsa-2023")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Rozsa et al. (2023) on DANDI:000341, part 1: slice recordings without induction groups

Paper: Rozsa, Toth, Olah, Baka, Lakovics, Barzo and Tamas (2023), "Temporal disparity of action
potentials triggered in axon initial segments and distal axons in the neocortex", Sci. Adv. 9,
eade4511, doi:10.1126/sciadv.ade4511. Claims file: `claims/rozsa-2023.yaml`, written from the
version of record (Europe PMC full text) and its supplement.

Dataset: DANDI:000341, version 0.241220.1312 (doi:10.48324/dandi.000341/0.241220.1312). It is the
only published version; the paper cites the dandiset without a version, so the most recent
published version is pinned. The draft has the same 787 assets and size.

This notebook covers the ZD7288 recordings (folder `Persistent_RAF_ZD7288`), the soma and bleb
recordings (`Axonal_bleb_recording`), the active human slices (`In_vitro_Human_elevated_neuronal_activity`)
and the active rat slices with layer 5 LFP (`In_vitro_oscillation_Rat`). The induction recordings
are in `reproduce_rozsa_2023_induction.ipynb`, the in vivo recordings in
`reproduce_rozsa_2023_invivo.ipynb`.

Revised after the first verification round: the threshold is found on a trace smoothed to the
Methods' 10 kHz filter, artefact APs are removed before any analysis, the mixture is fitted on
robustly scaled features and its components are printed, steady current is judged in absolute
terms, and the outputs the verifier asked for are added (metadata of each folder, Vm and LFP
clocks, bleb sweep pairing and traces, further readings for fig2c and fig2h).

| Claim | Here |
|---|---|
| fig2c-zd7288-reduces-raf | tested |
| figs2-zd7288-passive-properties | tested |
| fig1hk-bleb-initiation-order | tested |
| fig2h-bleb-sag-higher | tested |
| fig3-human-ap-states | tested |
| fig3f-human-pre-ap-fluctuation | tested |
| fig3i-rat-lfp-rhythm | tested |
| fig3l-rat-spontaneous-ra-proportion | tested |
| fig3k-rat-pre-ap-fluctuation | tested |
| fig3m-rat-ais-phase-coupling | tested |
| fig3m-rat-ra-no-phase | tested |
| figs3-no-modulators | tested |
| the induction claims (fig1ab, fig1c, fig1d, fig1f, figs1b, figs1d, fig1g, fig2e, fig2il) | other notebook |
| the in vivo claims (fig4*, figs4i, figs5) | in vivo notebook |

In [ ]:
import zlib, time, resource, collections
import numpy as np
import pandas as pd
from scipy import stats, signal
import matplotlib.pyplot as plt
import raf

%matplotlib inline
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
PAPER = "rozsa-2023"


def seed(claim):
    return zlib.crc32(f"{PAPER}/{claim}".encode())


T0 = time.time()
assets = raf.list_assets()
groups = collections.defaultdict(list)
for a in assets:
    groups[a["path"].split("/")[0]].append(a)
print(len(assets), "assets in", raf.DANDISET, raf.VERSION)
for g, v in groups.items():
    print(f"{g:45s} files {len(v):4d}  subjects {len({a['path'].split('/')[1] for a in v}):3d}  "
          f"{sum(a['size'] for a in v) / 1e9:7.1f} GB")

Each file is read whole into memory (parallel range requests) and reduced at once to a table of
APs and a table of sweeps (`raf.extract`); the raw traces are not kept, except the LFP of the rat
recordings (decimated to about 500 Hz) and the soma and bleb traces of the dual recordings. Artefact
APs (threshold outside -80 to -10 mV or peak above 100 mV, `raf.plausible`) are left out of every
AP analysis below.

In [ ]:
res = {}
for g in ("Persistent_RAF_ZD7288", "Axonal_bleb_recording", "In_vitro_Human_elevated_neuronal_activity",
          "In_vitro_oscillation_Rat"):
    t = time.time()
    keep = {"In_vitro_oscillation_Rat": ("LFP",), "Axonal_bleb_recording": ("Vm",)}.get(g, ())
    res[g] = raf.process_group(groups[g], keep=keep, workers=4)
    print(f"{g}: {len(res[g])} files in {time.time() - t:.0f} s; peak memory "
          f"{resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.1f} GB")


def table(g):
    """One row per file: metadata, APs (main detection rule) and sweeps."""
    rows = []
    for p, (m, aps, sw, lfp) in res[g].items():
        rows.append(dict(path=p, file=p.split("/")[-1], species=m["subject_species"],
                         subject=m["subject_subject_id"], age=m.get("subject_age"),
                         meta=m, aps=aps, sw=sw, lfp=lfp))
    return rows

## Data check

In [ ]:
for g in res:
    rows = table(g)
    print("==", g)
    print("  files", len(rows), "species", dict(collections.Counter(r["species"] for r in rows)),
          "subjects", len({r["subject"] for r in rows}))
    sw = pd.concat([r["sw"] for r in rows])
    print("  channels", dict(collections.Counter(sw.chan)))
    print("  sampling rates (Hz)", dict(collections.Counter(sw.rate.round())))
    vm = sw[sw.chan.str.startswith("Vm")]
    print(f"  Vm read as volts x 1000: median {vm.vmed.median():.1f} mV, range of sweep minima "
          f"{vm.vmin.min():.1f} to {vm.vmax.max():.1f} mV")
    print("  sweeps whose stimulus is stored in pA though its unit is amperes:",
          int(vm["stim_in_pA"].sum()) if "stim_in_pA" in vm else 0,
          "; with values above 1e16 pA, stimulus ignored:", int(vm["stim_bad"].sum()) if "stim_bad" in vm else 0)
    print(f"  stimulus relative to holding: {np.nanmin(vm.imin):.0f} to {np.nanmax(vm.imax):.0f} pA; "
          f"holding {np.nanmin(vm.hold):.0f} to {np.nanmax(vm.hold):.0f} pA")
    aps = pd.concat([r["aps"] for r in rows if len(r["aps"])])
    print(f"  AP candidates {len(aps)}, main rule (peak > 0 mV) {int(aps.main.sum())}, "
          f"variant (peak >= threshold + 30 mV) {int(aps.variant30.sum())}; main-rule artefacts left out "
          f"(threshold outside -80 to -10 mV or peak above 100 mV) {int((aps.main & ~raf.plausible(aps)).sum())}")
    m0 = rows[0]["meta"]
    print("  metadata of", rows[0]["file"])
    print("    general:", {k: v for k, v in m0["general"].items() if k not in ("pharmacology",)})
    print("    subject:", {k: v for k, v in m0.items() if k.startswith("subject_")})
    print("    electrodes:", m0["electrode_descriptions"])
    print("    pharmacology keys:", list(m0["pharmacology"]))
    print("    subject descriptions over the folder:",
          dict(collections.Counter(str(r["meta"].get("subject_description")) for r in rows).most_common(6)))
    ph = collections.Counter()
    for r in rows:
        for k, v in r["meta"]["pharmacology"].items():
            if v != "NaN" and ("Drug" in k and "Name" in k or k.startswith("Drugname")):
                ph[v] += 1
    print("  drugs named in general/pharmacology:", dict(ph))
    # Recordings deposited twice: the same sweep count and the same median Vm of the first 20 sweeps.
    fp = collections.defaultdict(list)
    for r in rows:
        v = r["sw"][r["sw"].chan.str.startswith("Vm")].sort_values("start")
        fp[(len(v), tuple(v.vmed.round(3).head(20)))].append(r["file"])
    dup = [v for v in fp.values() if len(v) > 1]
    print("  files with identical sweep fingerprints:", dup if dup else "none")
print()
print("The voltage series give conversion 1e-12 with unit volts, but the stored values are in volts")
print("(medians near -65 V x 1e-3 above); read with the stated conversion they would be picovolts.")

Clock check for the rat LFP recordings. Firing and LFP deflections are said to be synchronized
(Figure 3I), so on a shared clock the AP count in 1 s bins should be correlated with the LFP at
lag 0, and less so when the LFP is shifted by several seconds. The raw Vm is not kept, so the AP
times stand in for the up states.

In [ ]:
def vm_lfp_corr(r, shifts=(0, 2, 5, 10)):
    """Correlation of the LFP (1 s means) with the AP count in the same 1 s bins, averaged over
    LFP segments of at least 20 s with 5 APs or more, at LFP shifts in seconds."""
    if not r["lfp"] or not len(r["aps"]):
        return None
    out = {}
    t_ap = r["aps"][r["aps"].main].t_abs.to_numpy()
    for sh in shifts:
        cs = []
        for (st, rate, x) in r["lfp"].values():
            dur = len(x) / rate
            if dur < 20:
                continue
            nb = int(dur)
            lf = x[: nb * int(rate)].reshape(nb, int(rate)).mean(1)
            cnt = np.histogram(t_ap, bins=st + sh + np.arange(nb + 1))[0]
            if cnt.sum() >= 5 and np.std(lf) > 0:
                cs.append(np.corrcoef(lf, cnt)[0, 1])
        out[sh] = np.nanmean(cs) if cs else np.nan
    return out


rows_osc = table("In_vitro_oscillation_Rat")
cc = pd.DataFrame({r["file"]: vm_lfp_corr(r) for r in rows_osc}).T
print("Mean correlation of the 1 s LFP mean with the AP count, by LFP shift (s):")
print(cc.describe().round(3))

# Starting times of the Vm and LFP series side by side (first three recordings).
for r in rows_osc[:3]:
    s = r["sw"].sort_values("start")
    vm = s[s.chan.str.startswith("Vm")]
    lf = s[s.chan.str.startswith("LFP")]
    print(r["file"][:40])
    print("   Vm  (start s, duration s):", [(round(a, 2), round(b, 1)) for a, b in zip(vm.start.head(6), (vm.n / vm.rate).head(6))])
    print("   LFP (start s, duration s):", [(round(a, 2), round(b, 1)) for a, b in zip(lf.start.head(6), (lf.n / lf.rate).head(6))])
    same = len(set(vm.start.round(3)) & set(lf.start.round(3)))
    print(f"   Vm series {len(vm)}, LFP series {len(lf)}, sharing a starting time {same}")
# The same count over every recording with LFP series (verifier, round 2).
frac = []
for r in rows_osc:
    s = r["sw"]
    vm, lf = set(s[s.chan.str.startswith("Vm")].start.round(3)), set(s[s.chan.str.startswith("LFP")].start.round(3))
    if lf:
        frac.append(len(vm & lf) / len(lf))
print(f"recordings with LFP series {len(frac)}; LFP series sharing a starting time with a Vm series: median fraction "
      f"{np.median(frac):.3f}, lowest {min(frac):.3f}, recordings where every one does {sum(f == 1 for f in frac)}")


def ap_triggered_lfp(r, shift=0.0, half=1.0):
    """Mean LFP (about 500 Hz) from -1 to +1 s around the peaks of the APs of a recording (APs of the
    main rule, plausible), with the AP times shifted by `shift` seconds. Returns the mean, the number
    of APs and the LFP rate."""
    a = r["aps"]
    if not len(a) or not r["lfp"]:
        return None, 0, np.nan
    a = a[a.main & raf.plausible(a)]
    t = (a.sweep_start + a.t_peak).to_numpy() + shift
    segs = []
    for st, rate, x in r["lfp"].values():
        h = int(half * rate)
        i = np.round((t - st) * rate).astype(int)
        i = i[(i - h >= 0) & (i + h < len(x))]
        for k in i[:2000]:
            seg = x[k - h:k + h + 1].astype(float)
            segs.append(seg - seg.mean())
            used = rate
    if not segs:
        return None, 0, np.nan
    L = min(len(z) for z in segs)
    return np.mean([z[:L] for z in segs], axis=0), len(segs), used


trig = []
fig, ax = plt.subplots(figsize=(6, 3.5))
for r in rows_osc:
    m0, n0, fs0 = ap_triggered_lfp(r)
    if m0 is None or n0 < 20:
        continue
    c0, w0 = len(m0) // 2, int(round(0.05 * fs0))
    amp0 = m0[c0 - w0:c0 + w0 + 1].mean()
    null = [ap_triggered_lfp(r, shift=sh)[0] for sh in (-10, -5, 5, 10)]
    null = [z[len(z) // 2 - w0:len(z) // 2 + w0 + 1].mean() for z in null if z is not None]
    trig.append(dict(file=r["file"][:32], n_aps=n0, lfp_at_ap=amp0, shifted_mean=np.mean(null), shifted_sd=np.std(null),
                     lfp_sd=np.std(m0)))
    ax.plot((np.arange(len(m0)) - c0) / fs0 * 1000, m0, lw=0.6)
ax.set(xlabel="time from AP peak (ms)", ylabel="mean LFP (mV, segment mean removed)",
       title="AP-triggered LFP average, one line per recording")
plt.show()
trig = pd.DataFrame(trig)
print(trig.round(4).to_string())
z = (trig.lfp_at_ap - trig.shifted_mean) / trig.shifted_sd.replace(0, np.nan)
print(f"recordings {len(trig)}; mean LFP within 50 ms of the AP below the mean with AP times shifted by "
      f"5 or 10 s in {int((trig.lfp_at_ap < trig.shifted_mean).sum())}; |z| > 2 against the shifted values in "
      f"{int((z.abs() > 2).sum())}")

## fig2c-zd7288-reduces-raf

> Blocking HCN channels with ZD7288 (30 uM) reduces the duration of RAF (47.2 to 10.1 s) and the
> number of RA-APs (624 to 48) in human and rat layer 1 interneurons.

The wash-in time of ZD7288 is in each file's `general/pharmacology` (`DrugWashinTime1`, in
seconds on the file clock, checked below against the sweep times). Control is the part of the
recording before it, ZD7288 the part after. In each part, RA-APs by reading (c) of the claims
file's header (`raf.ra_aps_induction`): APs without positive current after the first induction
sweep of that part. RAF duration and RA-AP count as in fig1ab-raf-patterns, reading (i), one
epoch per part. The Methods count the RA-APs of the continuous recording ("Analysis of RAF
patterns": "number of RA-APs during continuous recording"); the APs in the interstimulus
intervals of the induction are counted in the main reading and left out in a variant. Epoch
reading (ii) of fig1ab, one epoch per induction run within each part, is shown too, with the
per-cell mean over the part's epochs (0 when it has none). The one file at 20 uM is left out in a
variant.

In [ ]:
def split_raf(aps, sw, t_from, t_to, continuous_only=False, reading="i"):
    a = aps[(aps.t_abs >= t_from) & (aps.t_abs < t_to)]
    s = sw[(sw.start >= t_from) & (sw.start < t_to)]
    ais, ra, t0, runs = raf.ra_aps_induction(a, s)
    if continuous_only and len(ra):
        stepless = set(s[s.nsteps == 0].series)
        ra = ra[ra.series.isin(stepless)]
    if reading == "i":
        ep = raf.epochs(ra.t_abs)
        dur, n = (ep[0]["duration"] if ep else 0.0), len(ra)
    else:
        ep = raf.epochs(ra.t_abs, runs) if len(runs) else []
        dur = float(np.mean([e["duration"] for e in ep])) if ep else 0.0
        n = float(np.mean([e["n"] for e in ep])) if ep else 0.0
    return dict(n_ra=n, duration=dur, n_epochs=len(ep), n_runs=len(runs),
                n_ind=len(raf.induction_sweeps(s)), t_ind=t0, ais=len(ais))


zd_rows = []
for r in table("Persistent_RAF_ZD7288"):
    ph = r["meta"]["pharmacology"]
    T = float(ph["DrugWashinTime1"])
    aps = r["aps"][r["aps"].main & raf.plausible(r["aps"])]
    sw = r["sw"]
    end = (sw.start + sw.n / sw.rate).max()
    for reading in ("i", "ii"):
        for cont in (False, True):
            c = split_raf(aps, sw, -np.inf, T, cont, reading)
            z = split_raf(aps, sw, T, np.inf, cont, reading)
            zd_rows.append(dict(file=r["file"], species=r["species"].split()[0], drug=ph["DrugName1"],
                                conc=ph["DrugConcentration1"], washin_s=T, rec_end_s=round(end, 1),
                                epochs=reading, continuous_only=cont,
                                ctrl_ind_sweeps=c["n_ind"], zd_ind_sweeps=z["n_ind"],
                                ctrl_runs=c["n_runs"], zd_runs=z["n_runs"], ctrl_epochs=c["n_epochs"], zd_epochs=z["n_epochs"],
                                ctrl_dur=c["duration"], zd_dur=z["duration"],
                                ctrl_n=c["n_ra"], zd_n=z["n_ra"]))
zd = pd.DataFrame(zd_rows)
with pd.option_context("display.width", 250):
    print(zd.drop(columns=["drug"]).round(2).to_string())

In [ ]:
EXP = {"dur": (47.2, 10.1), "n": (624, 48)}
for (reading, cont, only30), d in [((r_, c_, o_), d_[d_.conc.str.startswith("30")] if o_ else d_)
                                   for (r_, c_), d_ in zd.groupby(["epochs", "continuous_only"]) for o_ in (False, True)]:
    print(f"== epochs ({reading}), RA-APs {'of the continuous recording only' if cont else 'after induction, ISIs included'}"
          f"{', 30 uM files only' if only30 else ''} (n = {len(d)} cells)")
    hits = {"mean": 0, "median": 0}
    for q in ("dur", "n"):
        c, z = d[f"ctrl_{q}"], d[f"zd_{q}"]
        for stat in ("mean", "median"):
            hits[stat] += sum(abs(getattr(x, stat)() - e) <= 0.1 * e for x, e in zip((c, z), EXP[q]))
        c, z = d[f"ctrl_{q}"], d[f"zd_{q}"]
        p_t = stats.ttest_rel(c, z).pvalue
        rule = raf.paper_test(c, z, paired=True)
        print(f"  {q:4s} control mean {c.mean():8.1f} median {c.median():8.1f} | ZD7288 mean "
              f"{z.mean():8.1f} median {z.median():8.1f} | paired t P = {p_t:.4f}; "
              f"paper's rule: {rule[0]} P = {rule[1]:.4f}; cells decreasing {int((z < c).sum())}/{len(d)}")
    for sp, ds in d.groupby("species"):
        print(f"    {sp}: duration control {ds.ctrl_dur.round(1).tolist()} ZD {ds.zd_dur.round(1).tolist()}; "
              f"count control {ds.ctrl_n.round(1).tolist()} ZD {ds.zd_n.round(1).tolist()}")
    print(f"  values within tolerance: means {hits['mean']} of 4, medians {hits['median']} of 4")
print("expected: duration 47.2 -> 10.1 s, count 624 -> 48 (tolerance 10%), paired t P = 0.008 and 0.007")

**Result: not reproduced.** No reading gives both decreases at P < 0.05. With one epoch per part and the interstimulus intervals, RAF duration rises after the ZD7288 wash-in time (mean 131.6 to 359.7 s, paired t P = 0.017, falling in 1 of 7 cells) and the RA-AP count does not fall significantly (684.6 to 596.9, P = 0.83). With one epoch per induction run, duration 59.3 to 71.6 s (P = 0.45) and count 478.0 to 163.3 (P = 0.22). At most one of the four values is within tolerance under any reading. The three rat cells lose nearly all RA-APs (363, 915, 180 to 13, 13, 16).

## figs2-zd7288-passive-properties

> ZD7288 increases input resistance (70 to 98 MOhm, P = 0.01), and hyperpolarizes the resting
> potential (-60 to -67 mV, P = 0.075) and decreases sag fraction (0.04 to 0.017, P = 0.057)
> without reaching significance.

From the sweeps with a single hyperpolarizing step (`raf._sag`), before and after the wash-in
time. Input resistance: the steady deflection over the current of the smallest hyperpolarizing
step (criterion; standard choice); resting potential: the median of the pre-step baselines;
sag fraction as in fig2e-sag-species under reading (a) on the most hyperpolarizing step, with
reading (b) and the step whose peak is nearest -90 mV shown as variants. The pre-step baseline
includes any holding current; cells with a holding current in a period are listed.

In [ ]:
def passive(sw):
    s = sw.dropna(subset=["sag_a"]) if "sag_a" in sw else sw.iloc[:0]
    if not len(s):
        return dict(n=0)
    small = s[s.i_step == s.i_step.max()]
    big = s[s.i_step == s.i_step.min()]
    near = s.iloc[[int(np.argmin(np.abs(s.vpeak.to_numpy() + 90)))]]
    return dict(n=len(s), r_in=small.r_in.median(), v_rest=s.base.median(),
                hold=float(np.nanmedian(s.hold)),
                sag_a_big=big.sag_a.median(), sag_b_big=big.sag_b.median(),
                sag_a_90=near.sag_a.iloc[0], sag_b_90=near.sag_b.iloc[0],
                step_small=small.i_step.iloc[0], step_big=big.i_step.iloc[0])


pp = []
for r in table("Persistent_RAF_ZD7288"):
    T = float(r["meta"]["pharmacology"]["DrugWashinTime1"])
    sw = r["sw"][r["sw"].chan == "Vm"]
    for per, s in (("control", sw[sw.start < T]), ("ZD7288", sw[sw.start >= T])):
        pp.append(dict(file=r["file"], period=per, **passive(s)))
pp = pd.DataFrame(pp)
print(pp.round(3).to_string())

In [ ]:
w = pp.pivot(index="file", columns="period")
ok = w["n"].min(axis=1) > 0
print(f"cells with step sweeps in both periods: {int(ok.sum())} of {len(w)}")
w = w[ok]
for q, exp in (("r_in", (70, 98)), ("v_rest", (-60, -67)), ("sag_a_big", (0.04, 0.017)),
               ("sag_b_big", None), ("sag_a_90", None), ("sag_b_90", None)):
    c, z = w[q]["control"].astype(float), w[q]["ZD7288"].astype(float)
    m = ~(c.isna() | z.isna())
    c, z = c[m], z[m]
    p = stats.ttest_rel(c, z).pvalue if len(c) > 1 else np.nan
    rule = raf.paper_test(c, z, paired=True)
    print(f"{q:10s} n={len(c)} control median {c.median():8.3f} mean {c.mean():8.3f} | ZD7288 median "
          f"{z.median():8.3f} mean {z.mean():8.3f} | paired t P = {p:.3f}; paper's rule {rule[0]} P = {rule[1]:.3f}"
          + (f" | expected {exp[0]} -> {exp[1]}" if exp else ""))

**Result: not reproduced.** Input resistance does not rise (median 146.2 to 141.0 MOhm, paired t P = 0.78); resting potential (-63.6 to -63.4 mV) and sag fraction (0.056 to 0.047) do not change. Two of six values, the resting potentials, are within tolerance.

## fig1hk-bleb-initiation-order

> In dual soma and axonal bleb recordings of human NGFCs, AIS-APs reach the soma first and the
> bleb second, while RA-APs reach the bleb first, and RA-APs have lower somatic threshold and
> pre-AP slope.

The cell class is in each file's `general/pharmacology` (`celltype`: `NGF` or `no recovery`).
Somatic and bleb sweeps are paired by sweep number (`Vm_soma_<k>`, `Vm_bleb_<k>`; the pairs are
checked below to start at the same time). For each
somatic AP, the bleb AP within 3 ms is taken (a standard choice); latency is bleb time minus
soma time, at the threshold and at the peak (the criterion's two readings). AIS-APs and RA-APs by
reading (b), a Gaussian mixture on threshold and slope fitted to the somatic APs of the human
bleb recordings pooled; the dandiset has no AP labels, so reading (a) cannot be run. The bleb
threshold uses the same smoothed dV/dt as the somatic one (`raf.dvdt`).

The somatic thresholds of these files are checked against the voltage during the steps: an
unbalanced bridge adds the drop over the pipette resistance to the recorded potential during
current injection, which raises the apparent threshold of APs fired during depolarizing steps.
`jump` is the voltage step within 0.15 ms of the onset of a depolarizing step (`raf.extract`).
As a classification that does not rest on the threshold, the APs are also split by the sign of
their latency, and the two groups' thresholds and slopes compared.

In [ ]:
def bleb_latency(r, win_ms=3.0):
    """Soma-to-bleb latency for each somatic AP of a dual recording. The bleb signal is small (its
    peaks are mostly below 0 mV), so bleb APs are not detected on their own: in the bleb sweep of the
    same number, the bleb peak is the maximum within 3 ms of the somatic peak and the bleb threshold
    the first sample of the run with dV/dt >= 5 mV/ms before the steepest point ahead of that peak,
    as for somatic APs (standard choices). Latencies with no bleb run reaching 5 mV/ms are NaN."""
    aps = r["aps"]
    if not len(aps):
        return pd.DataFrame()
    so = aps[aps.main & raf.plausible(aps) & (aps.chan == "Vm_soma")].copy()
    raw = r["lfp"]
    out = []
    for _, a in so.iterrows():
        k = a.series.rsplit("_", 1)[1]
        b = raw.get(f"Vm_bleb_{k}")
        row = a.to_dict()
        if b is not None:
            st, rate, x = b
            ms = rate / 1000
            lo, hi = int((a.t_peak - win_ms / 1000) * rate), int((a.t_peak + win_ms / 1000) * rate)
            lo, hi = max(lo, 1), min(hi, len(x) - 1)
            if hi - lo > 4:
                seg = x[lo:hi].astype(float)
                pk = lo + int(np.argmax(seg))
                w = max(1, int(round(raf.SMOOTH_S * rate)))
                a0 = max(0, lo - 1 - w)
                dv = raf.dvdt(x[a0:pk + w + 1].astype(float), rate)[lo - 1 - a0:pk - a0]
                row["bleb_amp"] = float(x[pk] - np.median(x[max(0, lo - int(5 * ms)):lo]))
                row["lat_peak_ms"] = (pk / rate - a.t_peak) * 1000
                if len(dv) and dv.max() >= raf.DVDT:
                    j = int(np.argmax(dv))
                    while j > 0 and dv[j - 1] >= raf.DVDT:
                        j -= 1
                    row["lat_thr_ms"] = ((lo - 1 + j) / rate - a.t_thr) * 1000
                else:
                    row["lat_thr_ms"] = np.nan
        out.append(row)
    return pd.DataFrame(out)


bleb = []
for r in table("Axonal_bleb_recording"):
    ph = r["meta"]["pharmacology"]
    d = bleb_latency(r)
    n_bleb_sweeps = int((r["sw"].chan == "Vm_bleb").sum())
    bleb.append(dict(file=r["file"], species=r["species"].split()[0], celltype=ph.get("celltype"),
                     distance_um=float(ph["Electrode Distance (µm) xyz"]) if "Electrode Distance (µm) xyz" in ph else np.nan,
                     comment=ph.get("Comment", ""), bleb_sweeps=n_bleb_sweeps, pairs=d))
print(pd.DataFrame([{k: v for k, v in b.items() if k != "pairs"} | {
    "soma_APs": len(b["pairs"]), "with_bleb_sweep": int(b["pairs"].lat_peak_ms.notna().sum()) if "lat_peak_ms" in b["pairs"] else 0,
    "bleb_thr_found": int(b["pairs"].lat_thr_ms.notna().sum()) if "lat_thr_ms" in b["pairs"] else 0}
    for b in bleb]).to_string())
print()
print("Pairing and bridge check per file:")
for r in table("Axonal_bleb_recording"):
    sw = r["sw"]
    so, bl = sw[sw.chan == "Vm_soma"].copy(), sw[sw.chan == "Vm_bleb"].copy()
    for d in (so, bl):
        d["k"] = d.series.str.rsplit("_", n=1).str[1].astype(int)
    mm = so.merge(bl, on="k", suffixes=("_s", "_b"))
    dt = (mm.start_b - mm.start_s).abs()
    j = so.dropna(subset=["jump_mv"]) if "jump_mv" in so else so.iloc[:0]
    a = r["aps"]
    a = a[a.main & raf.plausible(a) & (a.chan == "Vm_soma")] if len(a) else a
    dep = a[a.i_thr > 5] if len(a) else a
    nodep = a[a.i_thr <= 5] if len(a) else a
    print(f"  {r['file'][:30]} soma sweeps {len(so)}, bleb {len(bl)}, pairs by number {len(mm)}, largest start "
          f"difference {dt.max() if len(dt) else np.nan:.4f} s; onset jump {np.median(j.jump_mv / j.jump_pa * 1000) if len(j) else np.nan:.0f} MOhm "
          f"(median over {len(j)} steps); somatic APs during depolarizing steps {len(dep)} (threshold median "
          f"{dep.thr.median() if len(dep) else np.nan:.1f} mV, current {dep.i_thr.median() if len(dep) else np.nan:.0f} pA), "
          f"without current {len(nodep)} (threshold median {nodep.thr.median() if len(nodep) else np.nan:.1f} mV); "
          f"pre-step baseline median {so.vmed.median():.1f} mV")

In [ ]:
hum = [b for b in bleb if b["species"] == "Homo" and len(b["pairs"])]
allh = pd.concat([b["pairs"].assign(file=b["file"], celltype=b["celltype"]) for b in hum])
allh["ra"], info = raf.gmm_classes(allh, seed("fig1hk-bleb-initiation-order"))
print(f"human somatic APs {len(allh)}; RA-APs by the mixture {int(allh.ra.sum())}; components",
      {k: (round(v, 2) if isinstance(v, float) else v) for k, v in info.items()})
print("RA component size under seeds +0, +1000 to +4000:", raf.gmm_seed_range(allh, seed("fig1hk-bleb-initiation-order")))
for (f, ct), d in allh.groupby(["file", "celltype"]):
    for cls, dd in d.groupby("ra"):
        name = "RA-AP " if cls else "AIS-AP"
        print(f"{f[:32]} {ct:12s} {name} n={len(dd):4d} with bleb sweep {int(dd.lat_peak_ms.notna().sum()) if 'lat_peak_ms' in dd else 0:4d} "
              f"median latency thr {dd.get('lat_thr_ms', pd.Series(dtype=float)).median():6.3f} ms "
              f"peak {dd.get('lat_peak_ms', pd.Series(dtype=float)).median():6.3f} ms | "
              f"threshold {dd.thr.median():6.1f} mV slope {dd.slope.median():5.2f} mV/ms")
# Rat recordings, for comparison (not judged)
rat = [b for b in bleb if b["species"] == "Rattus" and len(b["pairs"])]
allr = pd.concat([b["pairs"].assign(file=b["file"], celltype=b["celltype"]) for b in rat])
allr["ra"], _ = raf.gmm_classes(allr, seed("fig1hk-bleb-initiation-order") + 1)
print("rat, for comparison:")
for c in ("lat_thr_ms", "lat_peak_ms", "bleb_amp"):
    if c not in allr:
        allr[c] = np.nan
print(allr.groupby(["file", "celltype", "ra"]).agg(n=("thr", "size"), lat_thr=("lat_thr_ms", "median"),
                                       lat_peak=("lat_peak_ms", "median"), bleb_amp=("bleb_amp", "median"),
                                       thr=("thr", "median"), slope=("slope", "median")).round(3))
print()
print("Human NGFCs, APs split by the sign of the latency at threshold (bleb first: negative):")
for f, d in allh[allh.celltype == "NGF"].groupby("file"):
    d = d.dropna(subset=["lat_thr_ms"])
    for neg, dd in d.groupby(d.lat_thr_ms < 0):
        print(f"  {f[:30]} {'bleb first' if neg else 'soma first'}: n={len(dd)}, threshold median {dd.thr.median():.1f} mV, "
              f"slope {dd.slope.median():.2f} mV/ms, current at threshold median {dd.i_thr.median():.0f} pA, "
              f"latency at peak median {dd.lat_peak_ms.median():.2f} ms")

# Example soma and bleb traces of the two human NGFCs, thresholds marked.
raw_by_file = {r["file"]: r["lfp"] for r in table("Axonal_bleb_recording")}
ex = []
for f, d in allh[allh.celltype == "NGF"].groupby("file"):
    for cls, dd in d.dropna(subset=["lat_thr_ms"]).groupby("ra"):
        ex.append(dd.iloc[len(dd) // 2])
fig, axs = plt.subplots(1, len(ex), figsize=(4 * len(ex), 3.5), squeeze=False)
for ax, a in zip(axs[0], ex):
    raw = raw_by_file[a.file]
    st, rate, xs = raw[a.series]
    xb = raw[a.series.replace("soma", "bleb")][2]
    i0, h = int(a.thr_idx), int(0.004 * rate)
    t = (np.arange(i0 - h, i0 + h) - i0) / rate * 1000
    ax.plot(t, xs[i0 - h:i0 + h], "k", lw=0.8, label="soma")
    ax.plot(t, xb[i0 - h:i0 + h], "r", lw=0.8, label="bleb")
    ax.axvline(0, color="k", ls=":")
    if np.isfinite(a.lat_thr_ms):
        ax.axvline(a.lat_thr_ms, color="r", ls=":")
    ax.set(title=f"{a.file[4:16]} {a.series} {'RA' if a.ra else 'AIS'} class\nthr {a.thr:.1f} mV, I {a.i_thr:.0f} pA",
           xlabel="ms from somatic threshold", ylabel="mV")
    ax.legend(fontsize=7)
plt.tight_layout()
plt.show()
for a in ex:
    print(f"example {a.file[:30]} {a.series}: soma threshold {a.thr:.1f} mV at {a.t_thr:.4f} s, latency at threshold "
          f"{a.lat_thr_ms:.2f} ms, at peak {a.lat_peak_ms:.2f} ms")

**Result: not reproduced.** In the NGFC 220504 the bleb leads for the AIS class (median latency -0.20 ms at threshold, -0.14 ms at peak) as for the RA class (-0.18 ms at threshold); in 220505-2 all 62 somatic APs fall in the AIS class (+0.16 ms) and none in the RA class. The mixture's classes agree with the paper's description (RA component threshold -64.5 against -21.9 mV, slope 0.14 against 1.15 mV/ms). Split by the sign of the latency, the bleb-first APs of 220504 have the lower threshold (-27.5 against -18.3 mV) and slope (0.25 against 1.55 mV/ms), which is the paper's pattern, but that split is not one of the criterion's readings. The per-file median somatic thresholds lie at -30.4 to -11.2 mV, with or without current (the RA-class APs of 220504 and 220616 have medians of -60.6 and -69.6 mV), and the voltage jumps at step onset imply 26 to 57 MOhm of uncompensated resistance, so the somatic thresholds of these files are doubtful.

## fig2h-bleb-sag-higher

> Axonal blebs show about four times the sag fraction of the soma of the same layer 1
> interneurons (0.4 against 0.1), at 106 um Euclidean soma-bleb distance.

Each compartment's sag from the hyperpolarizing steps injected into it (soma trace with a
somatic step, bleb trace with a bleb step), per cell, under the claim's readings: formula (a) or
(b), on the most hyperpolarizing step or the step whose peak is nearest -90 mV. The criterion's
variant takes the bleb value from the bleb trace during a somatic step (`raf._sag` on the bleb
trace with the somatic step's timing). A two-sample t test across cells as the paper names it; a
paired t test as a variant. Checks asked for by the verifier: step sweeps whose sag peak lies below
-150 mV (an electrode artefact or bridge error; sweep minima reach -434 mV) are left out in a
variant, and the two files of one rat cell with two blebs (`2 bleb on same cell`) give one somatic
value in a variant. The distance is the file's `Electrode Distance (µm) xyz`.

In [ ]:
def sag_pick(s, how, clean=False):
    s = s.dropna(subset=["sag_a"]) if "sag_a" in s else s.iloc[:0]
    if clean and len(s):
        s = s[s.vpeak > -150]
    if not len(s):
        return None
    if how == "most":
        return s[s.i_step == s.i_step.min()].iloc[0]
    return s.iloc[int(np.argmin(np.abs(s.vpeak.to_numpy() + 90)))]


def bleb_during_soma(r):
    # Sag of the bleb trace during each somatic hyperpolarizing step sweep.
    out = []
    so = r["sw"][(r["sw"].chan == "Vm_soma")]
    so = so.dropna(subset=["sag_a"]) if "sag_a" in so else so.iloc[:0]
    for _, w in so.iterrows():
        b = r["lfp"].get(w.series.replace("soma", "bleb"))
        if b is None or len(w.neg_steps) != 1:
            continue
        st, rate, xb = b
        if len(xb) != w.n:
            continue
        sg_ = raf._sag(xb.astype(float), rate, list(w.neg_steps))
        if sg_:
            out.append(sg_)
    return pd.DataFrame(out)


rows = []
for r in table("Axonal_bleb_recording"):
    bs = bleb_during_soma(r)
    for how in ("most", "near90"):
        for clean in (False, True):
            so = sag_pick(r["sw"][r["sw"].chan == "Vm_soma"], how, clean)
            bl = sag_pick(r["sw"][r["sw"].chan == "Vm_bleb"], how, clean)
            bv = sag_pick(bs, how, clean)
            g = lambda x, k: getattr(x, k) if x is not None else np.nan
            rows.append(dict(file=r["file"][:30], species=r["species"].split()[0], step=how, clean=clean,
                             soma_a=g(so, "sag_a"), bleb_a=g(bl, "sag_a"), blebvar_a=g(bv, "sag_a"),
                             soma_b=g(so, "sag_b"), bleb_b=g(bl, "sag_b"), blebvar_b=g(bv, "sag_b"),
                             soma_step=g(so, "i_step"), bleb_step=g(bl, "i_step"), blebvar_step=g(bv, "i_step"),
                             soma_peak=g(so, "vpeak"), bleb_peak=g(bl, "vpeak"), blebvar_peak=g(bv, "vpeak")))
sg = pd.DataFrame(rows)
with pd.option_context("display.width", 250):
    print(sg[~sg.clean].drop(columns="clean").round(3).to_string())
TAB = []
for (how, clean), d in sg.groupby(["step", "clean"]):
    for f in ("a", "b"):
        for bleb_col in ("bleb", "blebvar"):
            for once in (False, True):
                dd = d.copy()
                if once:   # the second bleb of the same rat cell gives no second somatic value
                    dd.loc[dd.file.str.contains("2205111tm-22"), f"soma_{f}"] = np.nan
                s_, b_ = dd[f"soma_{f}"].dropna(), dd[f"{bleb_col}_{f}"].dropna()
                both = dd.dropna(subset=[f"soma_{f}", f"{bleb_col}_{f}"])
                p2 = stats.ttest_ind(b_, s_).pvalue if len(b_) > 1 and len(s_) > 1 else np.nan
                pp_ = stats.ttest_rel(both[f"{bleb_col}_{f}"], both[f"soma_{f}"]).pvalue if len(both) > 1 else np.nan
                TAB.append(dict(step=how, artefact_sweeps_out=clean, formula=f,
                                bleb_trace="bleb step" if bleb_col == "bleb" else "somatic step",
                                soma_once=once, soma_n=len(s_), soma_median=s_.median(), bleb_n=len(b_),
                                bleb_median=b_.median(), bleb_mean=b_.mean(), soma_mean=s_.mean(),
                                two_sample_P=p2, paired_P=pp_, n_pairs=len(both)))
TAB = pd.DataFrame(TAB)
with pd.option_context("display.width", 250, "display.max_rows", 100):
    print(TAB.round(4).to_string())
dist = pd.Series([b["distance_um"] for b in bleb])
print(f"soma-bleb distance: n={dist.notna().sum()} median {dist.median():.0f} um mean {dist.mean():.0f} um "
      f"range {dist.min():.0f} to {dist.max():.0f}")
print("expected: soma 0.1, bleb 0.4 (tolerance 10%), two-sample t P < 0.001, distance 106 um")

**Result: partially reproduced.** Under the criterion's main reading the bleb sag is higher than the somatic one (most hyperpolarizing step, formula (a): 0.115 against 0.024, two-sample t P = 0.0082; with the duplicated rat soma counted once, P = 0.011), but not at P < 0.001, and neither value is within tolerance. The difference rests on step sweeps whose sag peak lies below -150 mV: without them it vanishes (0.061 against 0.055, P = 0.28). The bleb trace during a somatic step gives values near 0.9 (P < 0.001), which reflects a bleb deflection too small for the formula. The median soma-bleb distance is 106 um, as in the paper.

## fig3-human-ap-states

> In active human slices (n = 26 cells), AIS-APs occur only in depolarized up states in all 7
> cells with AIS-APs, while spontaneous RA-APs, seen in 6 cells, also occur in hyperpolarized
> down and transition states.

APs of the sweeps without current injection only (the claim's "without current injection").
AIS-APs and RA-APs by reading (b): a Gaussian mixture on threshold and slope fitted to these APs
of the 26 human active-slice recordings pooled. The dandiset carries no AP labels (reading (a));
five files have a `Time To first RAAP (s)` entry in `general/pharmacology`, shown beside the
result. States per cell from the histogram of the membrane potential of the sweeps without
current, by the criterion's readings: (i) a two-Gaussian fit, down and up within one SD of each
mean, transition between; (ii) the range between the two means cut in thirds. The AP's
potential is the mean 0 to 50 ms before threshold, or the threshold.

In [ ]:
def no_current_aps(r):
    a, s = r["aps"], r["sw"]
    if not len(a):
        return a
    quiet = set(s[(s.nsteps == 0) & s.chan.str.startswith("Vm")].series)
    return a[a.main & raf.plausible(a) & a.series.isin(quiet)]


def states(hist, sd):
    """Two-Gaussian fit to a membrane potential histogram (bins raf.HIST_BINS); returns the bounds
    of down and up states under readings (i) and (ii)."""
    from sklearn.mixture import GaussianMixture
    centers = (raf.HIST_BINS[:-1] + raf.HIST_BINS[1:]) / 2
    w = np.round(hist / hist.sum() * 20000).astype(int)
    x = np.repeat(centers, w)[:, None]
    gm = GaussianMixture(2, random_state=sd).fit(x)
    order = np.argsort(gm.means_[:, 0])
    mu = gm.means_[order, 0]
    sdv = np.sqrt(gm.covariances_[order, 0, 0])
    i_down, i_up = mu[0] + sdv[0], mu[1] - sdv[1]
    if i_down > i_up:
        i_down = i_up = (mu[0] + mu[1]) / 2
    third = (mu[1] - mu[0]) / 3
    return dict(mu_down=mu[0], mu_up=mu[1], sd_down=sdv[0], sd_up=sdv[1],
                i_down=i_down, i_up=i_up, ii_down=mu[0] + third, ii_up=mu[1] - third)


def classify_state(v, b, reading):
    lo, hi = (b["i_down"], b["i_up"]) if reading == "i" else (b["ii_down"], b["ii_up"])
    return np.where(v <= lo, "down", np.where(v >= hi, "up", "transition"))


hrows = table("In_vitro_Human_elevated_neuronal_activity")
H = pd.concat([no_current_aps(r).assign(file=r["file"]) for r in hrows if len(no_current_aps(r))])
H["ra"], GM_H = raf.gmm_classes(H, seed("fig3-human-ap-states"))
print("mixture components:", {k: (round(v, 2) if isinstance(v, float) else v) for k, v in GM_H.items()})
print("RA component size under seeds +0, +1000 to +4000:", raf.gmm_seed_range(H, seed("fig3-human-ap-states")))
print(f"APs with a positive holding current (absolute current at threshold above 5 pA): {int((H.i_abs > 5).sum())}; "
      f"without a stimulus series: {int(H.i_abs.isna().sum())}")
fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(H.thr, H.slope.clip(-5, 15), s=3, c=np.where(H.ra, "r", "k"), alpha=0.4)
ax.set(xlabel="threshold (mV)", ylabel="pre-AP slope (mV/ms), clipped to -5 to 15",
       title="human active slices: red RA component, black AIS")
plt.show()
print(f"slopes outside -5 to 15 mV/ms: {int((~H.slope.between(-5, 15)).sum())}")
print(f"human active slices: {len(hrows)} files, {len(H)} APs without current, RA-APs {int(H.ra.sum())}")
print(H.groupby("ra")[["thr", "slope"]].describe().round(2).T)
bounds = {}
for r in hrows:
    s = r["sw"]
    hs = [h for h in s.get("hist", pd.Series(dtype=object)).dropna()] if "hist" in s else []
    if hs:
        bounds[r["file"]] = states(np.sum(hs, axis=0), seed("fig3-human-ap-states"))
print(pd.DataFrame(bounds).T.round(1).to_string())

In [ ]:
tf = {r["file"]: r["meta"]["pharmacology"].get("Time To first RAAP (s)", "NaN") for r in hrows}
res_states = {}
for reading in ("i", "ii"):
    for pot in ("vpre", "thr"):
        rows = []
        for f, d in H.groupby("file"):
            if f not in bounds:
                continue
            st = classify_state(d[pot].to_numpy(), bounds[f], reading)
            ais, ra = st[~d.ra.to_numpy()], st[d.ra.to_numpy()]
            rows.append(dict(file=f[:30], n_ais=len(ais), ais_up=int((ais == "up").sum()),
                             n_ra=len(ra), ra_down=int((ra == "down").sum()), ra_trans=int((ra == "transition").sum()),
                             ra_up=int((ra == "up").sum()), label_time_to_first_RAAP=tf[f]))
        t = pd.DataFrame(rows)
        res_states[(reading, pot)] = t
        n_ais = int((t.n_ais > 0).sum())
        all_up = int(((t.n_ais > 0) & (t.ais_up == t.n_ais)).sum())
        n_ra = int((t.n_ra > 0).sum())
        ra_dt = int(((t.n_ra > 0) & ((t.ra_down + t.ra_trans) > 0)).sum())
        print(f"reading ({reading}), potential {pot}: cells with AIS-APs {n_ais} (expected 7), all AIS-APs "
              f"in up states in {all_up}; cells with RA-APs {n_ra} (expected 6), with RA-APs in down or "
              f"transition states {ra_dt}")
print(res_states[("i", "vpre")].to_string())
print("files with a 'Time To first RAAP' entry:", {k[:30]: v for k, v in tf.items() if v != "NaN"})
d = H[H.file.isin([k for k, v in tf.items() if v != "NaN"])]
print("RA-APs by the mixture in those files:", d.groupby("file").ra.sum().to_dict())
print("variant AP rule (peak >= threshold + 30 mV): APs",
      int(sum(r["aps"].variant30.sum() for r in hrows if len(r["aps"]))), "of which also main",
      int(sum((r["aps"].variant30 & r["aps"].main).sum() for r in hrows if len(r["aps"]))))

**Result: blocked (methods_underspecified).** The mixture's lower-threshold component has the steeper mean slope (3.54 against 2.18 mV/ms), the opposite of the paper's RA-APs, so its classes are not the paper's and reading (a) cannot be run. For the record, with these classes 20 cells have AIS-APs and 12 RA-APs (expected 7 and 6); with the mean potential before threshold, reading (i), all AIS-APs are in up states in 10 of 20 cells and RA-APs reach down or transition states in 6 of 12.

## fig3f-human-pre-ap-fluctuation

> In active human slices, membrane potential fluctuations (SD) 0 to 50 ms before AIS-APs are
> larger than before RA-APs (1.49 against 0.78 mV).

The SD of the membrane potential in the 50 ms before each AP, the window ending at the threshold
or at the peak (the criterion's readings), APs pooled across cells, classes as above. APs with
less than 50 ms of trace before them are left out.

In [ ]:
def sd_compare(D, label):
    """Pooled and per-cell comparison; the variant 'isolated' keeps APs with no AP in the 50 ms
    before their threshold, so that the window holds no spike (not in the paper; shown because a
    preceding spike inflates the SD)."""
    D = D.sort_values(["file", "series", "t_thr"]).copy()
    prev = D.groupby(["file", "series"]).t_peak.shift(1)
    D["isolated"] = (D.t_thr - prev).fillna(np.inf) > 0.05
    for col in ("sd_thr", "sd_peak"):
        a, b = D[~D.ra & D.isolated][col].dropna(), D[D.ra & D.isolated][col].dropna()
        p = stats.mannwhitneyu(a, b).pvalue if len(a) and len(b) else np.nan
        print(f"{label} [isolated APs] window to {col[3:]}: AIS-AP {a.median():.2f} mV (n={len(a)}), "
              f"RA-AP {b.median():.2f} mV (n={len(b)}), Mann-Whitney P = {p:.2g}")
        a, b = D[~D.ra][col].dropna(), D[D.ra][col].dropna()
        p = stats.mannwhitneyu(a, b, alternative="two-sided").pvalue if len(a) and len(b) else np.nan
        pc = D.dropna(subset=[col]).groupby(["file", "ra"])[col].median().unstack()
        pcp = stats.mannwhitneyu(pc[False].dropna(), pc[True].dropna()).pvalue if True in pc and pc[True].notna().sum() > 0 else np.nan
        print(f"{label} window to {col[3:]}: AIS-AP SD median {a.median():.2f} mV (n={len(a)}), RA-AP "
              f"{b.median():.2f} mV (n={len(b)}), Mann-Whitney P = {p:.2g}; per-cell medians "
              f"P = {pcp:.2g} ({pc[False].notna().sum()} and {pc[True].notna().sum() if True in pc else 0} cells)")


sd_compare(H, "human")
print("expected: AIS 1.49 mV, RA 0.78 mV (tolerance 10%), Mann-Whitney P < 0.001")

**Result: blocked (methods_underspecified).** The classes are those of fig3-human-ap-states, whose RA component has the steeper slope. For the record, with these classes the SD before RA-APs is the higher one (window to threshold 5.43 against 3.18 mV, P = 2.2e-73; window to peak 7.13 against 4.97 mV).

## Rat active slices with layer 5 LFP

The condition of each recording is read from `general/pharmacology`: carbachol (also spelled
`Charbachol`) with SCH23390 is the active slice of Figure 3; recordings with neither are the
slices without modulators of Figure S3. Some recordings list further drugs (kainic acid,
gabazine, 4-AP); they are kept in the main set and left out in a variant. APs of the gap-free
sweeps of at least 5 s (`long_sweep_aps`), classed by reading (b), a mixture fitted to these APs
of all rat slice recordings of this folder pooled. The variant keeps only APs without steady
current: at most 5 pA at threshold in absolute terms, holding current included (the first round
measured it relative to the holding level, which removed nothing).

In [ ]:
def cond(m):
    drugs = {v.lower().replace("charbachol", "carbachol") for k, v in m["pharmacology"].items()
             if k.startswith("DrugName") and v != "NaN"}
    if {"carbachol", "sch23390"} <= drugs:
        return "modulators" + ("+" + "+".join(sorted(drugs - {"carbachol", "sch23390"})) if drugs - {"carbachol", "sch23390"} else "")
    if not drugs:
        return "none"
    return "+".join(sorted(drugs))


def long_sweep_aps(r, min_s=5.0):
    """APs of the gap-free sweeps (at least 5 s; a standard choice), with or without a steady
    current: the Methods say some cells needed steady somatic current to reach AIS-AP threshold
    ("In vitro oscillations")."""
    a, s = r["aps"], r["sw"]
    if not len(a):
        return a
    long_ = set(s[(s.n / s.rate >= min_s) & s.chan.str.startswith("Vm")].series)
    return a[a.main & raf.plausible(a) & a.series.isin(long_)]


R = []
for r in rows_osc:
    r["cond"] = cond(r["meta"])
    a = long_sweep_aps(r)
    if len(a):
        R.append(a.assign(file=r["file"], cond=r["cond"]))
R = pd.concat(R)
R["ra"], GM_R = raf.gmm_classes(R, seed("fig3l-rat-spontaneous-ra-proportion"))
R["steady"] = R.i_abs > 5
print("mixture components:", {k: (round(v, 2) if isinstance(v, float) else v) for k, v in GM_R.items()})
print("RA component size under seeds +0, +1000 to +4000:", raf.gmm_seed_range(R, seed("fig3l-rat-spontaneous-ra-proportion")))
print("recordings by condition:", dict(collections.Counter(r["cond"] for r in rows_osc)))
print(f"APs of gap-free sweeps {len(R)}, RA-APs {int(R.ra.sum())}; of these without steady current "
      f"(absolute current at most 5 pA) {int((~R.steady).sum())}, RA-APs among them {int((R.ra & ~R.steady).sum())}; "
      f"APs without a stimulus series {int(R.i_abs.isna().sum())}")
print(R.groupby("ra")[["thr", "slope"]].median().round(2))
fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(R.thr, R.slope.clip(-5, 15), s=2, c=np.where(R.ra, "b", "k"), alpha=0.3)
ax.set(xlabel="threshold (mV)", ylabel="pre-AP slope (mV/ms), clipped to -5 to 15",
       title="rat active slices: blue RA component, black AIS")
plt.show()
print(f"slopes outside -5 to 15 mV/ms: {int((~R.slope.between(-5, 15)).sum())}")

In [ ]:
# Share of APs in the RA class, against the paper's counts (verifier, round 2). Figure 3M counts 76
# RA-APs and 534 AIS-APs in 22 rat cells; the human bleb mixture of fig1hk is shown beside it.
per = R.groupby("file").ra.mean()
kept = R[R.file.isin([r["file"] for r in rows_osc if r["cond"].startswith("modulators")])]
print(f"rat active slices, all gap-free APs: RA class {int(R.ra.sum())} of {len(R)} ({R.ra.mean():.3f}); "
      f"carbachol + SCH23390 recordings {int(kept.ra.sum())} of {len(kept)} ({kept.ra.mean():.3f})")
print(f"   per recording, median RA share {per.median():.3f}; recordings where the RA class is more than half "
      f"of the APs {int((per > 0.5).sum())} of {len(per)}")
print(f"paper, Figure 3M: 76 of {76 + 534} APs ({76 / (76 + 534):.3f})")
print(f"human bleb mixture (fig1hk): {int(allh.ra.sum())} of {len(allh)} ({allh.ra.mean():.3f})")
print("component threshold medians (mV): RA", round(R[R.ra].thr.median(), 1), "AIS", round(R[~R.ra].thr.median(), 1))


The rat mixture passes the slope check, but its RA class holds about three quarters of the APs
(0.754 over all APs, 0.70 after the delta selection of the phase analysis below), against one
eighth in the paper's Figure 3M (76 of 610), and it is more than half of the APs in 28 of 32
recordings. Its classes therefore cannot be the paper's, which is what the human and in vivo
claims have in common with these (there the evidence was an inverted slope). The two classes also
lie 8 mV apart in threshold; in the rat induction recordings, where the classes come from the
protocol (reading (c)), RA-APs lie about 20 mV below AIS-APs (-61.6 against -41.2 mV in sporadic
cells, induction notebook, fig1c), but those AIS-APs are evoked by current pulses and at 100 kHz
the order reverses, so this comparison is weaker evidence than the share. The four rat claims that
rest on these classes (fig3l, fig3k, fig3m-rat-ais-phase-coupling, fig3m-rat-ra-no-phase) are
blocked (methods_underspecified); the verifier agreed in round 3.

## fig3i-rat-lfp-rhythm

> In active rat slices, layer 5 LFP shows rhythmic low-frequency deflections at 1.1 Hz,
> synchronized to up states of the layer 1 interneurons.

Per recording, LFP segments of at least 8 s, resampled to 500 Hz. Reading (i): the peak of the
Welch power spectrum (8 s windows, averaged over segments) in 0.5 to 4 Hz; reading (ii): the
number of cycles of the 0.5 to 4 Hz Butterworth-filtered LFP (order 2, zero phase, as in the
header) per second, counted as wraps of the Hilbert phase. Median over recordings.

In [ ]:
FS = 500.0


def lfp_segments(r, min_s=8.0):
    out = []
    for name, (st, rate, x) in sorted(r["lfp"].items(), key=lambda kv: kv[1][0]):
        if len(x) / rate < min_s:
            continue
        q = int(round(rate / FS))
        y = signal.decimate(x.astype(float), q, ftype="fir") if q > 1 else x.astype(float)
        out.append((st, rate / q, y))
    return out


def rhythm(r, band=(0.5, 4.0), order=2):
    segs = lfp_segments(r, 8.0 if band[0] >= 0.5 else 40.0)
    if not segs:
        return dict(n_seg=0, lfp_s=0.0)
    nper = int((8 if band[0] >= 0.5 else 40) * FS)
    ps = []
    cyc, dur = 0.0, 0.0
    sos = signal.butter(order, band, btype="band", fs=FS, output="sos")
    for st, fs, y in segs:
        f, p = signal.welch(y - y.mean(), fs=fs, nperseg=min(nper, len(y)))
        ps.append(np.interp(np.arange(0, 10, 0.025), f, p))
        ph = np.unwrap(np.angle(signal.hilbert(signal.sosfiltfilt(sos, y))))
        cyc += (ph[-1] - ph[0]) / (2 * np.pi)
        dur += len(y) / fs
    f = np.arange(0, 10, 0.025)
    P = np.mean(ps, axis=0)
    m = (f >= band[0]) & (f <= band[1])
    return dict(n_seg=len(segs), lfp_s=dur, peak_hz=f[m][np.argmax(P[m])], cycles_hz=cyc / dur)


rh = pd.DataFrame([dict(file=r["file"][:32], cond=r["cond"], **rhythm(r)) for r in rows_osc])
print(rh.round(3).to_string())

In [ ]:
for label, d in (("carbachol + SCH23390, all", rh[rh.cond.str.startswith("modulators")]),
                 ("carbachol + SCH23390 only", rh[rh.cond == "modulators"])):
    d = d[d.n_seg > 0]
    print(f"{label}: recordings with LFP {len(d)}; (i) spectral peak median {d.peak_hz.median():.2f} Hz "
          f"(SD {d.peak_hz.std():.2f}); (ii) cycle rate median {d.cycles_hz.median():.2f} Hz (SD {d.cycles_hz.std():.2f})")
print("expected: 1.1 Hz (tolerance 0.11)")

**Result: reproduced.** Over 36 recordings the LFP's spectral peak is at a median 1.06 Hz and its cycle rate 1.01 Hz (30 recordings without extra drugs: 1.19 and 1.03 Hz), all within 1.1 +- 0.11 Hz. The synchronization to up states is not judged (criterion).

## fig3l-rat-spontaneous-ra-proportion

> In active rat slices, 13 of 40 layer 1 interneurons fire spontaneous RA-APs, and all recovered
> ones (n = 5) are NGFCs.

In [ ]:
for label, keep, nosteady in (("carbachol + SCH23390, all", lambda c: c.startswith("modulators"), False),
                              ("carbachol + SCH23390 only", lambda c: c == "modulators", False),
                              ("carbachol + SCH23390, all, APs under steady current left out", lambda c: c.startswith("modulators"), True)):
    files = [r["file"] for r in rows_osc if keep(r["cond"])]
    d = R[R.file.isin(files)]
    if nosteady:
        d = d[~d.steady]
    per = d.groupby("file").ra.sum().reindex(files, fill_value=0)
    print(f"{label}: cells {len(files)}, with at least one RA-AP {int((per > 0).sum())}, with at least two "
          f"{int((per > 1).sum())}; RA-APs per cell {sorted(per[per > 0].astype(int).tolist())}")
print("expected: 13 cells (tolerance 1.3) of 40. Cell morphology: no cell type field in these files.")

**Result: blocked (methods_underspecified).** The mixture, whose RA component has the lower threshold and the lower slope as the paper describes (-50.8 against -44.1 mV, 1.12 against 2.79 mV/ms), labels RA-APs in 29 of 36 active-slice cells (23 of 30 without extra drugs; 28 of 36 without APs under steady current), against 13 of 40. The mixture's classes fail the share check above (about three quarters of the APs in the RA class, against one eighth in the paper), so these numbers do not test the claim; the outcome was changed from not reproduced to blocked after verification round 3.

## fig3k-rat-pre-ap-fluctuation

> In active rat slices, AIS-APs occur during up states with larger membrane potential
> fluctuations before them, and RA-APs during down states with smaller ones.

In [ ]:
RA_ = R[R.cond.str.startswith("modulators")]
sd_compare(RA_, "rat")
print("The state of each rat AP is not computed here: the rat files' sweeps without current are")
print("the same kind, but the claim does not judge it (criterion).")

**Result: blocked (methods_underspecified).** Under the mixture classes, the SD before AIS-APs is higher with the window ending at the peak (10.03 against 8.65 mV, Mann-Whitney P = 5.9e-75) and at threshold (8.78 against 7.82 mV, P = 4.3e-65), and among isolated APs; per-cell medians do not differ (P = 0.25 and 0.41). The mixture's classes fail the share check above (about three quarters of the APs in the RA class, against one eighth in the paper), so these numbers do not test the claim; the outcome was changed from reproduced to blocked after verification round 3.

## fig3m-rat-ais-phase-coupling and fig3m-rat-ra-no-phase

> In active rat slices, AIS-APs are coupled near the trough of the slow LFP oscillation (mean
> vector length 0.093, angle 314 degrees). RA-APs show no phase preference.

The Methods' phase procedure (header of the claims file): for each AP, the 4 s LFP segment
centred on its peak; FFT power times frequency; the AP is kept when the power in 0.5 to 4 Hz over
that in 0.5 to 45 Hz exceeds 0.1. The LFP segment is band-passed 0.5 to 4 Hz (Butterworth order
2, zero phase; order 4 the variant), Hilbert phase at the AP peak, shifted by 180 degrees so that
the trough is 0 and 360 degrees and the peak 180 (Methods convention). Rayleigh test on the APs
pooled across cells.

In [ ]:
def ap_phases(r, aps, order=2):
    out = []
    for name, (st, rate, x) in r["lfp"].items():
        dur = len(x) / rate
        sel = aps[(aps.t_peak_abs >= st + 2) & (aps.t_peak_abs <= st + dur - 2)]
        if not len(sel):
            continue
        q = int(round(rate / FS))
        y = signal.decimate(x.astype(float), q, ftype="fir") if q > 1 else x.astype(float)
        fs = rate / q
        sos = signal.butter(order, (0.5, 4.0), btype="band", fs=fs, output="sos")
        ph = np.angle(signal.hilbert(signal.sosfiltfilt(sos, y)), deg=True)
        for _, ap in sel.iterrows():
            i = int(round((ap.t_peak_abs - st) * fs))
            h = int(round(2 * fs))
            if i - h < 0 or i + h > len(y):
                continue
            seg = y[i - h:i + h]
            F = np.fft.rfftfreq(len(seg), 1 / fs)
            P = np.abs(np.fft.rfft(seg - seg.mean())) ** 2 * F
            ratio = P[(F >= 0.5) & (F <= 4)].sum() / P[(F >= 0.5) & (F <= 45)].sum()
            out.append(dict(file=r["file"], cond=r["cond"], ra=ap.ra, steady=ap.steady, ratio=ratio,
                            phase=(ph[i] + 180) % 360))
    return pd.DataFrame(out)


R["t_peak_abs"] = R.sweep_start + R.t_peak
PH = {}
for order in (2, 4):
    parts = []
    for r in rows_osc:
        if not r["cond"].startswith("modulators"):
            continue
        d = R[R.file == r["file"]]
        if len(d):
            parts.append(ap_phases(r, d, order))
    PH[order] = pd.concat(parts)
print({o: len(v) for o, v in PH.items()}, "APs with a 4 s LFP segment")

In [ ]:
def phase_report(P, label):
    for ra, name in ((False, "AIS-APs"), (True, "RA-APs")):
        d = P[P.ra == ra]
        k = d[d.ratio > 0.1]
        rr, ang, p, n = raf.rayleigh(k.phase)
        print(f"{label} {name}: with LFP {len(d)} from {d.file.nunique()} cells; kept by delta ratio {n} "
              f"from {k.file.nunique()} cells; vector length {rr:.3f}, angle {ang:.0f} deg, Rayleigh P = {p:.2g}")
        if len(k):
            print("   per cell (kept APs):", k.groupby("file").size().sort_values(ascending=False).head(25).to_dict())


phase_report(PH[2], "order 2")
phase_report(PH[4], "order 4")
phase_report(PH[2][PH[2].cond == "modulators"], "order 2, carbachol + SCH23390 only")
phase_report(PH[2][~PH[2].steady], "order 2, APs under steady current left out")
print("expected AIS-APs: length 0.093 (tolerance 0.0093), angle 314 deg (within 31.4), P < 0.001, n = 534 from 22 cells")
print("expected RA-APs: P = 0.07 (no preference at P < 0.05), n = 76 from 22 cells")

**Result: blocked (methods_underspecified).** Under the mixture classes and without the recordings with extra drugs, 230 AIS-APs from 10 cells are coupled near the trough (319 degrees, Rayleigh P = 2.4e-49) with a vector length of 0.655, not 0.093; with all 36 recordings the angle is 256 degrees, outside tolerance; order 4 gives 255 degrees. The mixture's classes fail the share check above (about three quarters of the APs in the RA class, against one eighth in the paper), so these numbers do not test the claim; the outcome was changed from partially reproduced to blocked after verification round 3.

In [ ]:
d = PH[2][PH[2].ra & (PH[2].ratio > 0.1)]
rr, ang, p, n = raf.rayleigh(d.phase)
print(f"RA-APs, order 2: n {n}, vector length {rr:.3f}, angle {ang:.0f} deg, Rayleigh P = {p:.3g}")
d2 = PH[2][PH[2].ra & (PH[2].ratio > 0.1) & (PH[2].cond == "modulators")]
print("order 2, carbachol + SCH23390 only: n %d, length %.3f, angle %.0f, P = %.3g" % (raf.rayleigh(d2.phase)[3], *raf.rayleigh(d2.phase)[:3]))
d4 = PH[4][PH[4].ra & (PH[4].ratio > 0.1)]
print("order 4: n %d, length %.3f, angle %.0f, P = %.3g" % (raf.rayleigh(d4.phase)[3], *raf.rayleigh(d4.phase)[:3]))
d5 = PH[2][PH[2].ra & (PH[2].ratio > 0.1) & ~PH[2].steady]
print("order 2, APs under steady current left out: n %d, length %.3f, angle %.0f, P = %.3g" % (raf.rayleigh(d5.phase)[3], *raf.rayleigh(d5.phase)[:3]))

**Result: blocked (methods_underspecified).** Under the mixture classes, RA-APs are phase coupled in every population (all recordings: 8896 APs, length 0.192, P = 1.3e-144; without extra drugs: 695 APs, length 0.575, P = 1.5e-110; APs under steady current left out: P = 6.3e-100). The mixture's classes fail the share check above (about three quarters of the APs in the RA class, against one eighth in the paper), so these numbers do not test the claim; the outcome was changed from not reproduced to blocked after verification round 3.

## figs3-no-modulators

> In rat slices in in vivo-like ACSF without carbachol and SCH23390 (n = 90), elevated network
> activity with low-frequency up states (0.11 Hz) appears and layer 1 interneurons fire
> spontaneous RA-APs (4 NGFCs and 5 non-NGFCs).

In [ ]:
nm = [r for r in rows_osc if r["cond"] == "none"]
print(f"recordings without carbachol and SCH23390: {len(nm)} (paper: 90)")
rows = []
for r in nm:
    x = rhythm(r, band=(0.05, 4.0))
    per = R[(R.file == r["file"])].ra.sum()
    rows.append(dict(file=r["file"][:32], ra_aps=int(per), **x))
d = pd.DataFrame(rows)
print(d.round(3).to_string())
if len(d):
    print(f"median spectral peak {d.peak_hz.median():.3f} Hz, cycle rate {d.cycles_hz.median():.3f} Hz; "
          f"cells with RA-APs {int((d.ra_aps > 0).sum())}")
print("expected: 0.11 Hz (tolerance 0.011), 9 cells with RA-APs")
print(f"notebook time {(time.time() - T0) / 60:.1f} min, peak memory "
      f"{resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.1f} GB")

**Result: blocked (data_missing).** Only 2 of the paper's 90 recordings without modulators are in the dandiset, both from one rat. Their spectral peak sits at the lower edge of the band (0.05 Hz), so no peak is found, and 2 cells cannot be compared with 9; the numbers above are shown for information.

## Summary

| Claim | Outcome |
|---|---|
| fig2c-zd7288-reduces-raf | not reproduced |
| figs2-zd7288-passive-properties | not reproduced |
| fig1hk-bleb-initiation-order | not reproduced |
| fig2h-bleb-sag-higher | partially reproduced |
| fig3-human-ap-states | blocked, methods_underspecified |
| fig3f-human-pre-ap-fluctuation | blocked, methods_underspecified |
| fig3i-rat-lfp-rhythm | reproduced |
| fig3l-rat-spontaneous-ra-proportion | blocked, methods_underspecified |
| fig3k-rat-pre-ap-fluctuation | blocked, methods_underspecified |
| fig3m-rat-ais-phase-coupling | blocked, methods_underspecified |
| fig3m-rat-ra-no-phase | blocked, methods_underspecified |
| figs3-no-modulators | blocked, data_missing |
| in vivo claims (9) | in vivo notebook |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the differences from the paper's method.

How the analysis differs from the paper's. The dandiset has no AP labels, so AIS-APs and RA-APs are separated by a two-component mixture on threshold and pre-AP slope (reading (b) of the claims file), fitted after artefact APs are removed. In the rat active slices and the human bleb recordings its RA component has the lower threshold and slope, as the paper describes; in the rat bleb recordings and the human active slices it has the steeper slope, so the two human claims are blocked. In the rat active slices the RA class holds 0.754 of the APs (0.70 after the delta selection), against 0.125 in the paper's Figure 3M, so its classes cannot be the paper's either, and the four claims on rat spontaneous activity that rest on them are blocked (methods_underspecified), as the verifier agreed in round 3. The claims file's reading (b) has no share check, so this is a judgement. The ZD7288 comparison splits each recording at the wash-in time the files give. The bleb latency uses the bleb trace around each somatic AP, since bleb APs are too small to detect on their own.

What was found about the dataset. The voltage series state a conversion of 1e-12 while the values are in volts; many stimulus series are stored in pA under a unit of amperes. Sampling rates range from 5 to 100 kHz in these folders. Cell classes are recorded only for the bleb recordings. One bleb file has no bleb sweeps; the per-file median somatic AP thresholds of the bleb recordings lie at -30.4 to -11.2 mV, and some of their hyperpolarizing steps reach below -150 mV. Only 2 of the 90 recordings without modulators are present, and several active rat recordings carry drugs the paper does not mention (gabazine, kainic acid, 4-AP). In 24 of the 39 rat recordings with an LFP every LFP series starts at the same time as a Vm series (median fraction 1.000, lowest 0.667). The AP count and the LFP are no more correlated at lag 0 than at shifted lags (mean -0.046), and the AP-triggered LFP average departs from its time-shifted value (|z| > 2) in 9 of 16 recordings, 7 below and 2 above it, so the evidence for a shared clock in the rat recordings is weak.